# Documentation: Multi-Vital Patient Simulator (v2.0)

## 1. Overview
This notebook serves as a **Stochastic Data Generator** for a Healthcare IoT pipeline. It simulates high-fidelity physiological telemetry for a single patient, generating exactly **100 data points** as individual JSON files. The generator is designed to test downstream **Medallion Architecture** logic, including Auto Loader ingestion, windowed aggregations, and real-time clinical alerting.

---

## 2. Core Logic & Methodology
The generator uses **Object-Oriented Programming (OOP)** to maintain a persistent state via the `ContinuousHeartMonitor` class, ensuring that each data point transitions logically from the previous reading.

### Physiological Modeling
* **State-Switching:** The patient transitions between three physiological states—`REST`, `EXERCISE`, and `CRISIS`—based on randomized probability thresholds.
* **Multi-Metric Correlation:** The simulation tracks four distinct vitals. Respiratory rate and body temperature are logically influenced by the current physiological state and heart rate.
* **Stochastic Autoregression (ARIMA-style):** Vitals are calculated using a drift-and-noise formula:
    $$V_{t} = V_{t-1} + \eta(Target - V_{t-1}) + \sigma$$
    * **Drift ($\eta$):** A percentage-based pull (e.g., 10% for BPM) toward the current state's target value.
    * **Volatility ($\sigma$):** Gaussian noise (Normal Distribution) to simulate natural biological variability and sensor "jitter."

---

## 3. Data Schema
The output is saved in JSON format with the following structure:

| Field | Type | Description |
| :--- | :--- | :--- |
| `patient_id` | String | Unique identifier (e.g., "PT-542") |
| `timestamp` | ISO-8601 | The wall-clock time of generation |
| `heart_rate_bpm` | Float | Simulated heart rate value |
| `respiratory_rate`| Float | Simulated breaths per minute |
| `spo2_percent` | Float | Blood oxygen saturation level |
| `body_temp_c` | Float | Body temperature in Celsius |
| `current_state` | String | Physiological state (`REST`, `EXERCISE`, `CRISIS`) |
| `sequence_id` | Integer | Incremental counter (1–100) for strict ordering |

---

## 4. Infrastructure & Storage
* **Catalog:** `patient_data`
* **Schema:** `ingestion_patient_vitals`
* **Storage Type:** Unity Catalog Volume (`raw_jsons`)
* **Path:** `/Volumes/patient_data/ingestion_patient_vitals/raw_jsons/`
* **File Naming:** `{patient_id}_seq{sequence_id:03d}_{timestamp}.json`

---

## 5. Usage Instructions
1.  **Environment:** Designed to run within a Databricks Notebook with access to Unity Catalog.
2.  **Automatic Cleanup:** Upon execution, the script attempts to clear the target Volume path using `dbutils.fs` to ensure a clean state for the simulation.
3.  **Execution:** The generator runs for 100 iterations with a **5-second delay** (`time.sleep(5)`) between files to simulate a steady stream of incoming IoT data.
4.  **Verification:** Monitor the standard output for real-time logs of BPM and State transitions every 20 records.

---

**Would you like me to help you write a Spark SQL query to read this data from the Volume into a Silver-level Delta table?**

# Data generator data

## Main Code

In [0]:
import json
import time
import random
import os
from datetime import datetime

# --- SETTINGS ---
CATALOG = "patient_data"
SCHEMA = "ingestion_patient_vitals"
VOLUME = "raw_jsons"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"

# --- CLEANUP FOR DATABRICKS ---
try:
    files = dbutils.fs.ls(volume_path)
    for file in files:
        dbutils.fs.rm(file.path)
    print(f"Cleanup Complete: Removed {len(files)} files.")
except Exception as e:
    print("Volume is already empty or does not exist.")

# Ensure volume exists
os.makedirs(VOLUME_PATH, exist_ok=True)

class ContinuousHeartMonitor:
    def __init__(self, patient_id):
        self.patient_id = patient_id
        self.bpm = 72.0       # Initial BPM
        self.resp_rate = 16.0  # New: Respiratory Rate
        self.spo2 = 98.0       # New: Oxygen Saturation
        self.temp = 37.0       # New: Temperature (Celsius)
        self.state = "REST"    # Start at rest
        self.tick_count = 0

    def update_bpm(self):
        """
        Simulates the logic of a State-Switching Model (like a Random Forest 
        deciding state) + ARIMA (for the smooth transition of values).
        """
        # 1. Randomly transition states to simulate a 'Day in the Life'
        roll = random.random()
        if roll < 0.005: self.state = "CRISIS"    # Rare event
        elif roll < 0.03: self.state = "EXERCISE" # Occasional activity
        elif roll < 0.10: self.state = "REST"     # Return to baseline

        # 2. Define target zones
        targets = {
            "REST": {"bpm": 70, "resp": 14, "spo2": 98, "temp": 37.0},
            "EXERCISE": {"bpm": 150, "resp": 30, "spo2": 97, "temp": 37.8},
            "CRISIS": {"bpm": 195, "resp": 40, "spo2": 88, "temp": 38.5}
        }
        
        target = targets[self.state]

        # 3. Drift and Noise (ARIMA-style)
        # Heart Rate
        self.bpm += (target["bpm"] - self.bpm) * 0.1 + random.normalvariate(0, 1.5)
        
        # Respiratory Rate (Correlated to BPM)
        self.resp_rate += (target["resp"] - self.resp_rate) * 0.08 + random.normalvariate(0, 0.5)
        
        # SpO2 (Stays high unless in CRISIS)
        self.spo2 += (target["spo2"] - self.spo2) * 0.05 + random.normalvariate(0, 0.2)
        
        # 4. Final Safety Bounds
        self.bpm = max(40, min(220, self.bpm))
        self.resp_rate = max(8, min(60, self.resp_rate))
        self.spo2 = max(70, min(100, self.spo2))
        
        self.tick_count += 1
        
        return {
            "patient_id": self.patient_id,
            "timestamp": datetime.now().isoformat(),
            "heart_rate_bpm": round(self.bpm, 2),
            "respiratory_rate": round(self.resp_rate, 1),
            "spo2_percent": round(self.spo2, 1),
            "body_temp_c": round(self.temp, 1),
            "current_state": self.state,
            "sequence_id": self.tick_count
        }

# --- EXECUTION --- #
import random
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

patient_profiles = ["PT-3","PT-1","PT-2","PT-158","PT-318","PT-671"]

selected_id = random.choice(patient_profiles) 

patient_sim = ContinuousHeartMonitor(selected_id)
total_points = 100

print(f"Generating {total_points} points for patient {patient_sim.patient_id}...")

try:
    # Changed from 'while True' to a fixed range
    for i in range(1, total_points + 1):
        data = patient_sim.update_bpm()
        
        # Save unique file for each record
        # Generate a clean timestamp string
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

        # New filename format
        file_name = f"{data['patient_id']}_seq{i:03d}_{timestamp_str}.json"
        full_path = os.path.join(VOLUME_PATH, file_name)
        
        with open(full_path, "w") as f:
            json.dump(data, f)
            
        if i % 20 == 0:
            print(f"Generated {i}/{total_points} | {data['heart_rate_bpm']} BPM | State: {data['current_state']}")
            
        # Small delay so it finishes in a few seconds
        time.sleep(5) 

except Exception as e:
    print(f"An error occurred: {e}")

print(f"\n--- Done! {total_points} points saved to {VOLUME_PATH} ---")